# Early Detection Evaluation

In [1]:
import os
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:


def evaluate_early_detection(
    reference_dpi_folder,
    evaluation_dpi_folder,
    ref_mask_name="transformed_stacked_masks.png",
    eval_mask_name="mask_after_morphology.png",
    threshold=127,
):
    """
    Pixel level evaluation of early sporulation detection.

    reference_dpi_folder
        root with one subfolder per sample
        each sample subfolder contains ref_mask_name

    evaluation_dpi_folder
        root with one subfolder per sample
        each sample subfolder contains eval_mask_name

    Both PNGs are binary masks
    white means sporulation pixel
    black means non sporulation pixel

    Returns
        results
            list of per sample dicts
        overall
            dict with aggregated counts and metrics
    """

    ref_root = Path(reference_dpi_folder)
    eval_root = Path(evaluation_dpi_folder)

    results = []

    for sample_dir in sorted(ref_root.iterdir()):
        if not sample_dir.is_dir():
            continue

        sample_name = sample_dir.name

        ref_path = sample_dir / ref_mask_name
        eval_path = eval_root / sample_name / eval_mask_name

        if not ref_path.exists() or not eval_path.exists():
            print(f"missing files for sample {sample_name}")
            continue

        ref_img = cv2.imread(str(ref_path), cv2.IMREAD_GRAYSCALE)
        eval_img = cv2.imread(str(eval_path), cv2.IMREAD_GRAYSCALE)

        if ref_img is None or eval_img is None:
            print(f"failed to read png for sample {sample_name}")
            continue

        if ref_img.shape != eval_img.shape:
            print(f"shape mismatch for sample {sample_name}: {ref_img.shape} vs {eval_img.shape}")
            continue

        ref_bin = ref_img > threshold
        eval_bin = eval_img > threshold

        # here leaf area is the entire image
        leaf_mask = np.ones_like(ref_bin, dtype=bool)

        TP = np.sum(eval_bin & ref_bin & leaf_mask)
        FP = np.sum(eval_bin & (~ref_bin) & leaf_mask)
        FN = np.sum((~eval_bin) & ref_bin & leaf_mask)
        TN = np.sum((~eval_bin) & (~ref_bin) & leaf_mask)

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        f1 = (
            2.0 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0.0
        )

        results.append(
            {
                "sample": sample_name,
                "TP": int(TP),
                "FP": int(FP),
                "FN": int(FN),
                "TN": int(TN),
                "precision": float(precision),
                "recall": float(recall),
                "f1": float(f1),
            }
        )

    if not results:
        return results, None

    # Calculate average precision, recall, and f1 per sample (only non-zero)
    precision_vals = [r["precision"] for r in results if r["precision"] > 0]
    recall_vals = [r["recall"] for r in results if r["recall"] > 0]
    f1_vals = [r["f1"] for r in results if r["f1"] > 0]

    precision_avg = np.mean(precision_vals) if precision_vals else 0.0
    recall_avg = np.mean(recall_vals) if recall_vals else 0.0
    f1_avg = np.mean(f1_vals) if f1_vals else 0.0

    TP_total = sum(r["TP"] for r in results)
    FP_total = sum(r["FP"] for r in results)
    FN_total = sum(r["FN"] for r in results)
    TN_total = sum(r["TN"] for r in results)

    overall = {
        "TP_total": int(TP_total),
        "FP_total": int(FP_total),
        "FN_total": int(FN_total),
        "TN_total": int(TN_total),
        "precision_avg": float(precision_avg),
        "recall_avg": float(recall_avg),
        "f1_avg": float(f1_avg),
    }

    return results, overall

In [ ]:
ref_folder = "path_to_reference_dpi_folder"
eval_folder = "path_to_evaluation_dpi_folder"

results, overall = evaluate_early_detection(ref_folder, eval_folder)
print(overall)